In [1]:

"""
PI-DeepONet for 1D linear advection on a periodic domain with:
  - branch input = sampled initial condition at sensor points
  - hard initial condition enforcement
  - richer periodic Fourier trunk
  - decoder fusion (not just branch-trunk dot product)

PDE:
    u_t + u_x = 0,     (x,t) in [0,1] x [0,1]

Initial condition family:
    u(x,0) = exp( - d_per(x, x0)^2 / (2 nu^2) )

No characteristic-based features are used.
"""

import os
import csv
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR


# ============================================================
# Device and reproducibility
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Random seed set to:", SEED)

FIG_DIR = "figures"
DATA_DIR = "outputs_pideeponet"
MODEL_PATH = "pideeponet_advection.pt"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ============================================================
# Problem setup
# ============================================================
a_adv = 1.0
x0_min, x0_max = 0.35, 0.65
nu_min, nu_max = 0.03, 0.12

N_SENSORS = 80
x_sensors = np.linspace(0.0, 1.0, N_SENSORS, endpoint=False).astype(np.float32)
x_sensors_t = torch.tensor(x_sensors, dtype=torch.float32, device=device).view(-1)


def periodic_distance_np(x, c):
    d = np.abs(x - c)
    return np.minimum(d, 1.0 - d)


def periodic_distance_torch(x, c):
    d = torch.abs(x - c)
    return torch.minimum(d, 1.0 - d)


def periodic_gaussian_np(x, x0, nu):
    d = periodic_distance_np(x, x0)
    return np.exp(-(d ** 2) / (2.0 * nu ** 2))


def periodic_gaussian_torch(x, x0, nu):
    d = periodic_distance_torch(x, x0)
    return torch.exp(-(d ** 2) / (2.0 * nu ** 2))


def exact_solution_np(x, t, x0, nu, a=a_adv):
    xc = (x0 + a * t) % 1.0
    return periodic_gaussian_np(x, xc, nu)


def sample_pde_param(nu_min_curr=None, nu_max_curr=None):
    if nu_min_curr is None:
        nu_min_curr = nu_min
    if nu_max_curr is None:
        nu_max_curr = nu_max

    x0 = x0_min + (x0_max - x0_min) * torch.rand(1, device=device)
    u = torch.rand(1, device=device)
    nu_min_t = torch.tensor(nu_min_curr, device=device)
    nu_max_t = torch.tensor(nu_max_curr, device=device)
    nu = 10.0 ** (
        torch.log10(nu_min_t)
        + (torch.log10(nu_max_t) - torch.log10(nu_min_t)) * u
    )
    return torch.stack([x0, nu], dim=-1)  # (1,2)


def build_branch_input_from_p(p):
    x0, nu = p[0, 0], p[0, 1]
    return periodic_gaussian_torch(x_sensors_t, x0, nu).view(1, -1)


# ============================================================
# Sampling
# ============================================================
def sample_training_points(N_int, N_bc, N_near_ic, device=device):
    x_int = torch.rand(N_int, 1, device=device)
    t_int = torch.rand(N_int, 1, device=device)
    XT_int = torch.cat([x_int, t_int], dim=1)

    x_nic = torch.rand(N_near_ic, 1, device=device)
    t_nic = 0.25 * (torch.rand(N_near_ic, 1, device=device) ** 2)
    XT_near_ic = torch.cat([x_nic, t_nic], dim=1)

    t_bc = torch.rand(N_bc, 1, device=device)
    XT_left = torch.cat([torch.zeros(N_bc, 1, device=device), t_bc], dim=1)
    XT_right = torch.cat([torch.ones(N_bc, 1, device=device), t_bc], dim=1)

    return XT_int, XT_near_ic, XT_left, XT_right


# ============================================================
# Feature maps and MLPs
# ============================================================
class RichPeriodicFeatureTransform(nn.Module):
    """
    Rich periodic features in x and simple multiscale time features.
    """
    def __init__(self, n_fourier_x=12, n_fourier_t=4):
        super().__init__()
        self.n_fourier_x = n_fourier_x
        self.n_fourier_t = n_fourier_t

    def forward(self, xt):
        x = xt[:, 0:1]
        t = xt[:, 1:2]
        feats = [t, t * t]
        for k in range(1, self.n_fourier_x + 1):
            feats.append(torch.cos(2.0 * math.pi * k * x))
            feats.append(torch.sin(2.0 * math.pi * k * x))
        for k in range(1, self.n_fourier_t + 1):
            feats.append(torch.cos(math.pi * k * t))
            feats.append(torch.sin(math.pi * k * t))
        return torch.cat(feats, dim=1)


class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, depth=3, activation=nn.Tanh):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden_dim), activation()]
            d = hidden_dim
        layers += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ============================================================
# Model
# ============================================================
class HardICPIDeepONet(nn.Module):
    """
    u(x,t;v) = u0_rec(x; v) + t * correction(x,t;v)

    where:
      - u0_rec(x;v) reconstructs the IC from sensor-sampled branch input
      - correction is built from branch + trunk + decoder
      - periodicity in x is represented through Fourier features
    """
    def __init__(
        self,
        n_sensors=80,
        latent_dim=128,
        hidden_dim=128,
        depth_branch=3,
        depth_trunk=3,
        depth_decoder=2,
        n_fourier_x=12,
        n_fourier_t=4,
    ):
        super().__init__()
        self.feature_map = RichPeriodicFeatureTransform(
            n_fourier_x=n_fourier_x,
            n_fourier_t=n_fourier_t,
        )

        feat_dim = 2 + 2 * n_fourier_x + 2 * n_fourier_t

        self.branch = MLP(n_sensors, hidden_dim, latent_dim, depth=depth_branch, activation=nn.Tanh)
        self.trunk = MLP(feat_dim, hidden_dim, latent_dim, depth=depth_trunk, activation=nn.Tanh)
        self.decoder = MLP(3 * latent_dim, hidden_dim, 1, depth=depth_decoder, activation=nn.Tanh)

        # Separate IC reconstruction branch
        self.ic_branch = MLP(n_sensors, hidden_dim, latent_dim, depth=depth_branch, activation=nn.Tanh)
        self.ic_trunk = MLP(2 * n_fourier_x, hidden_dim, latent_dim, depth=depth_trunk, activation=nn.Tanh)
        self.ic_bias = nn.Parameter(torch.zeros(1))

        self.n_fourier_x = n_fourier_x

    def x_periodic_features(self, x):
        feats = []
        for k in range(1, self.n_fourier_x + 1):
            feats.append(torch.cos(2.0 * math.pi * k * x))
            feats.append(torch.sin(2.0 * math.pi * k * x))
        return torch.cat(feats, dim=1)

    def reconstruct_ic(self, v_branch, x):
        b0 = self.ic_branch(v_branch)            # (1,L)
        tx = self.ic_trunk(self.x_periodic_features(x))   # (N,L)
        return torch.sum(tx * b0, dim=1, keepdim=True) + self.ic_bias.view(1, 1)

    def forward(self, v_branch, xt):
        assert v_branch.shape[0] == 1, "This implementation assumes one branch input per call."

        x = xt[:, 0:1]
        t = xt[:, 1:2]

        # Hard IC anchor
        u0 = self.reconstruct_ic(v_branch, x)

        # Correction
        b = self.branch(v_branch)                # (1,L)
        phi = self.feature_map(xt)               # (N,F)
        tr = self.trunk(phi)                     # (N,L)
        b_rep = b.repeat(tr.shape[0], 1)

        fused = torch.cat([tr, b_rep, tr * b_rep], dim=1)
        corr = self.decoder(fused)               # (N,1)

        return u0 + t * corr


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================
# Differentiation helpers
# ============================================================
def advection_residual(model, v_branch, xt):
    xt = xt.clone().detach().requires_grad_(True)
    u = model(v_branch, xt)
    grads = torch.autograd.grad(u, xt, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = grads[:, 0:1]
    u_t = grads[:, 1:2]
    return u_t + a_adv * u_x


def spatial_derivative(model, v_branch, xt):
    xt = xt.clone().detach().requires_grad_(True)
    u = model(v_branch, xt)
    grads = torch.autograd.grad(u, xt, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    return grads[:, 0:1]


# ============================================================
# Evaluation
# ============================================================
def evaluate_on_grid(model, xg, tg, x0, nu):
    X, T = np.meshgrid(xg, tg, indexing="ij")
    XT = np.stack([X.reshape(-1), T.reshape(-1)], axis=1)
    XT_t = torch.tensor(XT, dtype=torch.float32, device=device)

    p = torch.tensor([[x0, nu]], dtype=torch.float32, device=device)
    v_branch = build_branch_input_from_p(p)

    model.eval()
    with torch.no_grad():
        u_pred = model(v_branch, XT_t).cpu().numpy().reshape(len(xg), len(tg))

    u_exact = exact_solution_np(X, T, x0, nu)
    abs_err = np.abs(u_pred - u_exact)
    rel_l2 = np.linalg.norm((u_pred - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-12)
    return u_exact, u_pred, abs_err, rel_l2


def save_summary_csv(rows, path):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["case_name", "x0", "nu", "rel_l2"])
        writer.writerows(rows)
    print(f"Saved summary CSV: {path}")


# ============================================================
# Training
# ============================================================
def train_model(
    epochs=3000,
    lr=5e-4,
    tasks_per_batch=8,
    N_int=2048,
    N_bc=256,
    N_near_ic=1024,
    latent_dim=128,
    hidden_dim=128,
):
    model = HardICPIDeepONet(
        n_sensors=N_SENSORS,
        latent_dim=latent_dim,
        hidden_dim=hidden_dim,
        depth_branch=3,
        depth_trunk=3,
        depth_decoder=2,
        n_fourier_x=12,
        n_fourier_t=4,
    ).to(device)

    print(f"Model params: {count_params(model):,d}")

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    scheduler = StepLR(optimizer, step_size=max(epochs // 2, 1), gamma=0.5)

    best_loss = float("inf")
    best_state = None
    history = {"epoch": [], "total": [], "bc": [], "res": [], "icrec": [], "lr": []}

    for ep in range(1, epochs + 1):
        optimizer.zero_grad()

        total_loss = 0.0
        total_bc = 0.0
        total_res = 0.0
        total_icrec = 0.0

        if ep <= epochs // 2:
            nu_min_curr, nu_max_curr = max(0.06, nu_min), nu_max
        else:
            nu_min_curr, nu_max_curr = nu_min, nu_max

        for _ in range(tasks_per_batch):
            p = sample_pde_param(nu_min_curr, nu_max_curr)
            v_branch = build_branch_input_from_p(p)

            XT_int, XT_near_ic, XT_left, XT_right = sample_training_points(
                N_int=N_int,
                N_bc=N_bc,
                N_near_ic=N_near_ic,
                device=device,
            )

            x0, nu = p[0, 0], p[0, 1]

            # residual
            res_int = advection_residual(model, v_branch, XT_int)
            res_nic = advection_residual(model, v_branch, XT_near_ic)
            loss_res = torch.mean(res_int ** 2) + torch.mean(res_nic ** 2)

            # periodic BC value + derivative (mostly for monitoring / soft stabilization)
            uL = model(v_branch, XT_left)
            uR = model(v_branch, XT_right)
            uxL = spatial_derivative(model, v_branch, XT_left)
            uxR = spatial_derivative(model, v_branch, XT_right)
            loss_bc = torch.mean((uL - uR) ** 2) + torch.mean((uxL - uxR) ** 2)

            # auxiliary IC reconstruction loss on random x points
            x_aux = torch.rand(512, 1, device=device)
            u0_rec = model.reconstruct_ic(v_branch, x_aux)
            u0_true = periodic_gaussian_torch(x_aux[:, 0], x0, nu).view(-1, 1)
            loss_icrec = torch.mean((u0_rec - u0_true) ** 2)

            # Emphasize IC reconstruction early so hard IC is meaningful
            lambda_icrec = 20.0 if ep <= epochs // 3 else 5.0

            loss_task = loss_res + 0.1 * loss_bc + lambda_icrec * loss_icrec

            total_loss += loss_task
            total_bc += loss_bc
            total_res += loss_res
            total_icrec += loss_icrec

        total_loss = total_loss / tasks_per_batch
        total_bc = total_bc / tasks_per_batch
        total_res = total_res / tasks_per_batch
        total_icrec = total_icrec / tasks_per_batch

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        lr_now = scheduler.get_last_lr()[0]
        history["epoch"].append(float(ep))
        history["total"].append(float(total_loss.item()))
        history["bc"].append(float(total_bc.item()))
        history["res"].append(float(total_res.item()))
        history["icrec"].append(float(total_icrec.item()))
        history["lr"].append(float(lr_now))

        if total_loss.item() < best_loss:
            best_loss = total_loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep == 1 or ep % 200 == 0:
            print(
                f"Epoch {ep:5d} | total={total_loss.item():.3e} | "
                f"res={total_res.item():.3e} | bc={total_bc.item():.3e} | "
                f"icrec={total_icrec.item():.3e} | lr={lr_now:.1e}"
            )

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"Best loss = {best_loss:.3e}")
    return model, history


# ============================================================
# Plotting / export
# ============================================================
test_cases = [
    ("center_in_range", 0.50, 0.07),
    ("left_in_range",   0.42, 0.09),
    ("shifted_oor",     0.25, 0.06),
    ("narrow_oor",      0.50, 0.025),
]


def plot_training_curves(history):
    plt.figure(figsize=(6, 4))
    plt.plot(history["epoch"], history["total"], label="Total")
    plt.plot(history["epoch"], history["res"], label="Residual", linestyle="--")
    plt.plot(history["epoch"], history["icrec"], label="IC reconstruction", linestyle=":")
    plt.plot(history["epoch"], history["bc"], label="BC", linestyle="-.")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training losses (PI-DeepONet)")
    plt.grid(True)
    plt.legend()
    out = os.path.join(FIG_DIR, "pideeponet_training.png")
    plt.savefig(out, dpi=250, bbox_inches="tight")
    print("Saved:", out)
    plt.close()


def evaluate_and_export(model, Nx=160, Nt=120, out_dir=DATA_DIR):
    os.makedirs(out_dir, exist_ok=True)

    xg = np.linspace(0.0, 1.0, Nx, endpoint=False)
    tg = np.linspace(0.0, 1.0, Nt)

    n_cases = len(test_cases)
    fig, axs = plt.subplots(n_cases, 3, figsize=(15, 4 * n_cases), constrained_layout=True)
    if n_cases == 1:
        axs = np.array([axs])

    summary_rows = []
    all_u_exact = []
    all_u_pred = []
    all_abs_err = []
    all_params = []
    all_names = []

    for i, (name, x0, nu) in enumerate(test_cases):
        u_exact, u_pred, abs_err, rel_l2 = evaluate_on_grid(model, xg, tg, x0, nu)
        print(f"{name:16s} | relL2 = {rel_l2:.3e}")

        summary_rows.append([name, x0, nu, rel_l2])
        all_u_exact.append(u_exact.astype(np.float32))
        all_u_pred.append(u_pred.astype(np.float32))
        all_abs_err.append(abs_err.astype(np.float32))
        all_params.append([x0, nu])
        all_names.append(name)

        per_case_path = os.path.join(out_dir, f"pideeponet_case_{i:02d}_{name}.npz")
        np.savez_compressed(
            per_case_path,
            x=xg.astype(np.float32),
            t=tg.astype(np.float32),
            params=np.asarray([x0, nu], dtype=np.float32),
            case_name=np.asarray(name),
            u_exact=u_exact.astype(np.float32),
            u_pred=u_pred.astype(np.float32),
            abs_error=abs_err.astype(np.float32),
            rel_l2=np.asarray(rel_l2, dtype=np.float32),
            method=np.asarray("PI-DeepONet"),
        )

        ax0, ax1, ax2 = axs[i, 0], axs[i, 1], axs[i, 2]
        im0 = ax0.imshow(u_exact.T, origin="lower", aspect="auto", extent=[0, 1, 0, 1])
        ax0.set_title("Exact")
        ax0.set_xlabel("x")
        ax0.set_ylabel("t")
        plt.colorbar(im0, ax=ax0)

        im1 = ax1.imshow(u_pred.T, origin="lower", aspect="auto", extent=[0, 1, 0, 1])
        ax1.set_title("PI-DeepONet pred")
        ax1.set_xlabel("x")
        ax1.set_ylabel("t")
        plt.colorbar(im1, ax=ax1)

        im2 = ax2.imshow(abs_err.T, origin="lower", aspect="auto", extent=[0, 1, 0, 1])
        ax2.set_title("|error|")
        ax2.set_xlabel("x")
        ax2.set_ylabel("t")
        plt.colorbar(im2, ax=ax2)

    fig_path = os.path.join(FIG_DIR, "pideeponet_results.png")
    plt.savefig(fig_path, dpi=250, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.close()

    bundle_path = os.path.join(out_dir, "pideeponet_bundle.npz")
    np.savez_compressed(
        bundle_path,
        x=xg.astype(np.float32),
        t=tg.astype(np.float32),
        params=np.asarray(all_params, dtype=np.float32),
        case_names=np.asarray(all_names),
        u_exact=np.stack(all_u_exact, axis=0),
        u_pred=np.stack(all_u_pred, axis=0),
        abs_error=np.stack(all_abs_err, axis=0),
        method=np.asarray("PI-DeepONet"),
    )
    print("Saved standardized bundle:", bundle_path)

    csv_path = os.path.join(out_dir, "pideeponet_summary.csv")
    save_summary_csv(summary_rows, csv_path)


if __name__ == "__main__":
    model, history = train_model(
        epochs=3000,
        lr=5e-4,
        tasks_per_batch=8,
        N_int=2048,
        N_bc=256,
        N_near_ic=1024,
        latent_dim=128,
        hidden_dim=128,
    )

    torch.save(model.state_dict(), MODEL_PATH)
    print("Saved model:", MODEL_PATH)

    plot_training_curves(history)
    evaluate_and_export(model, Nx=160, Nt=120, out_dir=DATA_DIR)

Using device: cpu
Random seed set to: 1234
Model params: 292,482
Epoch     1 | total=1.166e+01 | res=8.519e+00 | bc=2.170e-12 | icrec=1.571e-01 | lr=5.0e-04
Epoch   200 | total=1.529e+00 | res=3.949e-01 | bc=5.267e-13 | icrec=5.670e-02 | lr=5.0e-04
Epoch   400 | total=5.513e-01 | res=2.597e-01 | bc=7.184e-12 | icrec=1.458e-02 | lr=5.0e-04
Epoch   600 | total=3.068e-01 | res=1.821e-01 | bc=7.939e-12 | icrec=6.234e-03 | lr=5.0e-04
Epoch   800 | total=2.821e-01 | res=2.032e-01 | bc=8.512e-12 | icrec=3.943e-03 | lr=5.0e-04
Epoch  1000 | total=1.942e-01 | res=1.056e-01 | bc=7.800e-12 | icrec=4.429e-03 | lr=5.0e-04
Epoch  1200 | total=1.451e-01 | res=1.006e-01 | bc=8.218e-12 | icrec=8.884e-03 | lr=5.0e-04
Epoch  1400 | total=8.802e-02 | res=3.724e-02 | bc=4.185e-12 | icrec=1.016e-02 | lr=5.0e-04
Epoch  1600 | total=1.569e-01 | res=6.828e-02 | bc=4.353e-12 | icrec=1.773e-02 | lr=2.5e-04
Epoch  1800 | total=1.125e-01 | res=3.495e-02 | bc=4.555e-12 | icrec=1.552e-02 | lr=2.5e-04
Epoch  2000 | t